![Cabecera](assets/cabecera_thebridge.png)

# Proveedores de IA — Práctica en directo

Notebook de la sesión en clase. Acompaña a la guía **`guia_proveedores_ia.html`** — ahí están los conceptos explicados con calma; aquí solo **ejecutáis y observáis** (el código ya viene muy comentado línea a línea, para que entendáis qué hace cada parte, no solo que "funcione").

**Plan de hoy (intercalado, ~1h):**

| Bloque | Qué hacemos |
|---|---|
| 🟣 1. Cohere | Concepto en la guía → ejecutar esta sección |
| 🟠 2. Tour Hugging Face | Sin código — exploráis la plataforma en el navegador (individual) |
| 🟣 3. HF en local | Concepto en la guía → ejecutar esta sección |
| ✅ Cierre | Recapitulación y comparativa |
| 🎁 Bonus | Streaming y HF en la nube, si sobra tiempo |

## Antes de empezar

- [ ] Cuenta y **API key de Cohere** creada (ver guía, sección Cohere → "Cómo obtener la API key")

Si no trajiste la API key de Cohere: podéis sacarla ahora en 2 minutos en [dashboard.cohere.com](https://dashboard.cohere.com/) → *API Keys* → *Trial key*, mientras el resto del grupo sigue adelante.

---
## 🟣 Bloque 1 — Cohere (chat en la nube)

Si ya usasteis la API de **Gemini**, esto os va a sonar muy familiar: cliente + clave + lista de mensajes + parámetros. Cambia el nombre de las clases y algún detalle de formato, pero la lógica es la misma. Por eso aquí los comentarios son más ligeros — id a la guía si necesitáis el repaso conceptual completo.

### Instalación

Descomenta la celda siguiente la primera vez:

In [ ]:
# %pip install cohere

### Configura tu clave

In [1]:
import getpass

# getpass oculta lo que escribes en la terminal/celda (igual que con GEMINI_API_KEY):
# así la clave no queda visible en pantalla ni en el historial del notebook.
COHERE_API_KEY = getpass.getpass("Pega tu COHERE_API_KEY (input oculto): ").strip()
print("COHERE_API_KEY configurada:", "sí" if COHERE_API_KEY else "no")

COHERE_API_KEY configurada: sí


### Ejemplo 1 — una pregunta, una respuesta

Ejecuta y observa la respuesta.

In [2]:
import cohere

if not COHERE_API_KEY:
    raise ValueError("Falta COHERE_API_KEY. Ejecuta la celda anterior e introduce la clave.")

# Igual que con Gemini elegíais un modelo (gemini-1.5-flash, gemini-2.0-...), aquí elegís
# la familia Command de Cohere. "plus" = más potente, más lento; la versión sin "plus" es más ligera.
MODELO_COHERE = "command-r-plus-08-2024"  # alternativa ligera: command-r-08-2024
TEMPERATURE = 0.2  # mismo concepto que en Gemini: 0 = respuestas predecibles, 1 = más variadas

# El cliente es el equivalente al genai.Client(api_key=...) de Gemini: el objeto que
# guarda tu clave y sabe hablar con la API. "V2" es simplemente la versión actual del SDK.
client = cohere.ClientV2(api_key=COHERE_API_KEY)

response = client.chat(
    model=MODELO_COHERE,
    # "messages" es el mismo concepto que "contents" en Gemini: una lista de turnos.
    # Cada turno es un diccionario con "role" (quién habla) y "content" (qué dice).
    messages=[
        {"role": "user", "content": "En una frase: ¿qué es un asistente de onboarding para empleados nuevos?"}
    ],
    temperature=TEMPERATURE,
)

# La respuesta no es un string directo (a diferencia de response.text en Gemini):
# es un objeto con .message.content, que es una LISTA de bloques (por si hubiera varias
# partes, como texto + citas). Para chat simple, el texto está en la posición [0].
print(response.message.content[0].text)

Un asistente de onboarding es un profesional encargado de facilitar la integración y capacitación de los empleados nuevos en una empresa, asegurando una transición efectiva y una experiencia positiva durante su incorporación al puesto de trabajo.


### Ejemplo 2 — rol de sistema + dominio Bridge SA

Mismo patrón que usaréis en `prompts.py` del Team Challenge: **system prompt fijo** + pregunta del usuario.

In [3]:
# El system prompt cumple la misma función que el "system_instruction" de Gemini:
# se manda una sola vez y fija el comportamiento del asistente para toda la conversación.
# Triple comillas porque es un texto largo de varias líneas; .strip() quita saltos de línea
# sobrantes al principio/final.
SYSTEM_PROMPT = """
Eres el Employee Onboarding Assistant de Bridge SA.
Solo ayudas a empleados nuevos con accesos, políticas y primeros días.
No inventes políticas. Responde en español, breve.
""".strip()

pregunta_empleado = "¿A qué canales de Slack debo unirme el primer día?"

response = client.chat(
    model=MODELO_COHERE,
    messages=[
        # El orden importa: "system" siempre va primero, luego el turno "user".
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pregunta_empleado},
    ],
    temperature=TEMPERATURE,
)

print(response.message.content[0].text)

El primer día, es recomendable que te unas a los siguientes canales de Slack:

- #general: para anuncios y actualizaciones importantes de la empresa.
- #bienvenida: para dar la bienvenida a los nuevos empleados y hacer preguntas generales.
- #equipo-[tu equipo]: únete al canal específico de tu equipo para comunicarte con tus compañeros.
- #ayuda-it: para solicitar asistencia técnica o de TI.
- #proyectos: si es relevante para tu rol, únete a los canales de proyectos específicos para estar al tanto de las discusiones y actualizaciones.

Puedes explorar otros canales según tus intereses y responsabilidades a medida que te familiarices con la plataforma.


### Ejemplo 3 — conversación con memoria (varios turnos)

El modelo solo "recuerda" lo que va en `messages`. Cada turno se añade a la lista antes de la siguiente llamada — igual que con `chat.send_message()` en Gemini, pero aquí lo hacemos a mano.

In [4]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Soy Laura, dev junior. Hoy es mi día 1 de onboarding."},
]

# Primera llamada: el modelo responde solo con lo que hay en "messages" hasta ahora.
r1 = client.chat(model=MODELO_COHERE, messages=messages, temperature=TEMPERATURE)

# Clave para tener memoria: añadimos la respuesta del modelo a la lista con role "assistant",
# ANTES de mandar la siguiente pregunta. Si no la añadierais, el modelo no sabría de qué
# se habló en el turno anterior — cada llamada a la API es independiente por sí sola.
messages.append({"role": "assistant", "content": r1.message.content[0].text})

messages.append({"role": "user", "content": "¿Y qué hago con GitHub antes de clonar repos?"})

# Segunda llamada: ahora "messages" contiene los 3 turnos anteriores + esta pregunta,
# así que el modelo tiene contexto completo de la conversación.
r2 = client.chat(model=MODELO_COHERE, messages=messages, temperature=TEMPERATURE)
print(r2.message.content[0].text)

Antes de clonar repositorios, asegúrate de tener acceso a la cuenta de GitHub de la empresa y de conocer las políticas de uso y colaboración. Consulta con el equipo de desarrollo o tu mentor para obtener más información sobre los flujos de trabajo y las prácticas de la empresa.


### 🧪 Ahora tú

Cambia la pregunta por una tuya (relacionada con onboarding o con tu propio Team Challenge) y vuelve a ejecutar.

In [ ]:
pregunta_libre = "TODO: escribe aquí tu propia pregunta"

response = client.chat(
    model=MODELO_COHERE,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pregunta_libre},
    ],
    temperature=TEMPERATURE,
)

print(response.message.content[0].text)

---
## 🟠 Bloque 2 — Tour por Hugging Face (sin código)

Aquí cerráis el notebook un momento. Abrid **`guia_proveedores_ia.html`**, id a la sección **"Tour Hugging Face"** y completad — cada uno por su cuenta — la **misión de exploración** (checklist interactiva, ~5 min) directamente en el navegador.

Cuando volváis, deberíais poder responder sin mirar:

### Autocomprobación

- ¿Qué es el **Hub** de Hugging Face, en una frase?
- ¿Qué hace el **router** cuando no indicáis `provider=`?
- ¿Qué significa que un modelo sea **gated**?

Si alguna no la tenéis clara, es el momento de preguntar antes de seguir con la parte local.

---
## 🟣 Bloque 3 — Hugging Face en local

Esto **sí** es distinto a todo lo que habéis usado hasta ahora (Gemini, Cohere): no hay API, no hay servidor externo — el modelo se descarga y corre en vuestra propia máquina. Por eso aquí el código viene comentado con mucho más detalle: cada línea importa para entender qué está pasando.

### Instalación

Descomenta la celda siguiente la primera vez:

In [6]:
# transformers: librería de Hugging Face para cargar y ejecutar modelos.
# torch: PyTorch, el motor de cálculo numérico sobre el que corre el modelo (tensores, GPU, etc.).
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


### ¿Qué tienes ya en caché?

Antes de descargar nada, miramos qué modelos ya están guardados en tu disco de una vez anterior.

In [1]:
from huggingface_hub import scan_cache_dir

# scan_cache_dir() mira dentro de ~/.cache/huggingface/hub/ (la carpeta donde se
# guardan los pesos ya descargados) y lista qué modelos hay y cuánto espacio ocupan.
# No lista nada que esté "instalado" como un programa: son solo archivos de pesos.
#
# Si es la PRIMERA vez que usas Hugging Face en esta máquina, esa carpeta todavía
# no existe y scan_cache_dir() lanza un error: lo capturamos y avisamos, es normal.
try:
    info = scan_cache_dir()
    if info.repos:
        for repo in info.repos:
            print(repo.repo_id, f"({repo.size_on_disk_str})")
    else:
        print("La caché existe pero está vacía todavía.")
except Exception:
    print("Aún no hay caché de Hugging Face en esta máquina (normal la primera vez). Se creará sola al descargar el modelo en el siguiente paso.")

c:\_aTrabajo\_Projects\TheBridge\individual\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen/Qwen2.5-0.5B-Instruct (11.5M)


### Cargar el modelo

`EJECUTAR_LOCAL = True` ya está activado para la práctica de hoy: la primera vez descarga ~1 GB, tarda un par de minutos. Si tu máquina va muy justa de espacio o de RAM, cámbialo a `False` y sigue en modo lectura.

**Lee los comentarios de esta celda con calma** — es la parte más distinta a lo que ya conocéis de las APIs en la nube.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# AutoTokenizer: convierte texto <-> números (tokens) que el modelo puede procesar.
# AutoModelForCausalLM: la clase que sabe cargar y ejecutar un modelo "generador de texto"
#   (causal = predice la siguiente palabra a partir de las anteriores, como GPT, Llama, Qwen...).
# "Auto" en ambos nombres significa que transformers detecta solas la arquitectura correcta
# a partir del MODELO_LOCAL que le paséis, sin que tengáis que saber los detalles internos.

EJECUTAR_LOCAL = True  # cámbialo a False si prefieres no descargar el modelo hoy
MODELO_LOCAL = "Qwen/Qwen2.5-0.5B-Instruct"  # formato "autor/nombre-repo" del Hub
MAX_NEW_TOKENS = 150  # cuántos tokens como máximo puede generar el modelo (a más, más tarda)
TEMPERATURE_LOCAL = 0.2  # mismo concepto que en Cohere/Gemini: aleatoriedad de la generación

tokenizer_local = None  # se rellenará más abajo, si EJECUTAR_LOCAL = True
model_local = None

# A diferencia de la nube (donde el proveedor decide en qué máquina corre el modelo),
# aquí SOMOS NOSOTROS quienes elegimos el "device": GPU (cuda) si hay una disponible
# y compatible, si no, CPU. En un portátil normal, casi siempre será "cpu".
device_local = "cuda" if torch.cuda.is_available() else "cpu"


def chat_local(messages: list[dict], max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Genera una respuesta con el modelo local a partir de una lista de mensajes
    con el mismo formato role/content que ya usasteis con Cohere y Gemini."""
    if model_local is None or tokenizer_local is None:
        raise RuntimeError("Modelo local no cargado. Ejecuta la celda con EJECUTAR_LOCAL = True.")

    # 1) En la nube, el proveedor convierte tu lista de messages en el formato interno
    #    que su modelo espera SIN que lo veáis. En local, esa conversión la hacemos
    #    nosotros a mano con apply_chat_template: cada familia de modelos (Qwen, Llama,
    #    Mistral...) tiene su propia plantilla de texto para marcar dónde empieza el
    #    system, dónde el user, dónde debe responder el assistant, etc.
    prompt = tokenizer_local.apply_chat_template(
        messages,
        tokenize=False,          # de momento queremos ver el resultado como texto, no como números
        add_generation_prompt=True,  # añade la marca especial que le dice al modelo "ahora responde tú"
    )

    # 2) Ahora sí: convertimos ese texto a tensores (arrays de números) con el tokenizer,
    #    y los movemos al mismo "device" (CPU/GPU) donde está cargado el modelo.
    #    return_tensors="pt" = formato PyTorch (de ahí el "pt").
    inputs = tokenizer_local(prompt, return_tensors="pt").to(device_local)

    # 3) torch.no_grad() desactiva el cálculo de gradientes: solo estamos generando texto,
    #    no entrenando el modelo, así que ahorramos memoria y tiempo.
    with torch.no_grad():
        output_ids = model_local.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,              # sin esto, temperature no tendría ningún efecto
            temperature=TEMPERATURE_LOCAL,
            pad_token_id=tokenizer_local.eos_token_id,  # evita un warning; usa el token de "fin" como relleno
        )

    # 4) output_ids incluye el prompt de entrada + la respuesta generada, todo junto.
    #    Nos quedamos solo con los tokens NUEVOS (los que vienen después de la entrada),
    #    para no imprimir la pregunta otra vez dentro de la respuesta.
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]

    # 5) decode() convierte esos números de vuelta a texto legible.
    #    skip_special_tokens=True quita marcas internas como <|end|> que no queremos ver.
    return tokenizer_local.decode(new_tokens, skip_special_tokens=True).strip()


if EJECUTAR_LOCAL:
    print(f"Cargando {MODELO_LOCAL} en {device_local} (primera vez: descarga ~1 GB)...")

    # from_pretrained() busca el modelo en tu caché local; si no está, lo descarga del Hub
    # la primera vez y lo deja guardado para las siguientes ejecuciones.
    tokenizer_local = AutoTokenizer.from_pretrained(MODELO_LOCAL)
    model_local = AutoModelForCausalLM.from_pretrained(
        MODELO_LOCAL,
        # dtype = precisión numérica de los pesos del modelo.
        # float32 en CPU (más compatible); float16 en GPU (ocupa la mitad de memoria).
        dtype=torch.float32 if device_local == "cpu" else torch.float16,
    ).to(device_local)  # mueve el modelo entero a RAM (CPU) o VRAM (GPU)

    model_local.eval()  # modo "evaluación": apaga comportamientos de entrenamiento (p. ej. dropout)
    print("Modelo local listo.")
else:
    print("HF local omitido (EJECUTAR_LOCAL = False). Cambia a True para probar.")

Cargando Qwen/Qwen2.5-0.5B-Instruct en cpu (primera vez: descarga ~1 GB)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1086.51it/s]


Modelo local listo.


### Ejemplo 1 — una pregunta

La misma pregunta que usamos con Cohere, para poder comparar directamente el resultado.

In [3]:
if EJECUTAR_LOCAL:
    # chat_local espera la misma estructura de messages que ya conocéis (role/content).
    # Toda la complejidad de plantillas y tensores queda escondida dentro de la función.
    respuesta_local = chat_local([
        {"role": "user", "content": "En una frase: ¿qué es un asistente de onboarding para empleados nuevos?"}
    ])
    print(f"Modelo: {MODELO_LOCAL} ({device_local})")
    print(respuesta_local)
else:
    print("Activa EJECUTAR_LOCAL en la celda anterior.")

Modelo: Qwen/Qwen2.5-0.5B-Instruct (cpu)
Un asistente de onboarding para empleados nuevos es un recurso integral que ayuda a los nuevos empleados a familiarizarse con la organización y sus procesos operativos. Este tipo de asistencia se enfoca en proporcionar información relevante sobre el entorno laboral, las políticas y normas, así como instrucciones sobre cómo inicia su nueva jornada en la empresa.

El objetivo principal del asistente de onboarding es facilitar la integración de los nuevos empleados en la estructura organizacional, asegurando que estén al tanto de los principios básicos de la empresa y puedan comenzar su día a día sin complicaciones. Esto puede incluir:

1. **Información


### Ejemplo 2 — Bridge SA (mismo system prompt que en Cohere)

In [ ]:
SYSTEM_PROMPT_LOCAL = """
Eres el Employee Onboarding Assistant de Bridge SA.
Solo ayudas a empleados nuevos con accesos, políticas y primeros días.
No inventes políticas. Responde en español, breve.
""".strip()

if EJECUTAR_LOCAL:
    # Igual que con Cohere: "system" primero, "user" después. apply_chat_template()
    # se encarga de traducir esto al formato interno de Qwen.
    respuesta_bridge = chat_local([
        {"role": "system", "content": SYSTEM_PROMPT_LOCAL},
        {"role": "user", "content": "¿A qué canales de Slack debo unirme el primer día?"},
    ])
    print(respuesta_bridge)
else:
    print("Activa EJECUTAR_LOCAL en la celda de carga del modelo.")

### 🧪 Ahora tú

Misma pregunta libre que en el Bloque 1, pero ahora con el modelo local. Compara el tono y la calidad de la respuesta con la de Cohere — es un modelo mucho más pequeño (0,5B parámetros frente a los decenas de miles de millones de un modelo en la nube), así que se nota.

In [ ]:
pregunta_libre_local = "TODO: escribe aquí tu propia pregunta"

if EJECUTAR_LOCAL:
    respuesta = chat_local([
        {"role": "system", "content": SYSTEM_PROMPT_LOCAL},
        {"role": "user", "content": pregunta_libre_local},
    ])
    print(respuesta)
else:
    print("Activa EJECUTAR_LOCAL en la celda de carga del modelo.")

---
## ✅ Cierre — recapitulación

| | Cohere (nube) | HF local |
|---|---|---|
| API key | Sí | No |
| Velocidad | Rápida | Lenta en CPU |
| Privacidad | Datos viajan al proveedor | Todo en tu máquina |
| Ideal para | Prototipo empresarial rápido | Prototipar sin cuota ni conexión |

Repaso completo, con el bloque de Hugging Face en la nube incluido, en la sección **"Comparativa"** y **"¿Cuál elijo?"** de `guia_proveedores_ia.html`.

**Para vuestro repo:** si elegís un proveedor distinto de Gemini, la idea es la misma en los tres casos — un cliente dedicado (`cohere_client.py`, `hf_client.py`, `hf_local_client.py`) con una función `safe_generate(prompt, ...)` que devuelva `(texto, métricas)`. `prompts.py`, `logic.py` y `validators.py` no cambian: solo sustituís la capa de llamada a la API.

---
## 🎁 Bonus opcional

### Streaming con Cohere

Respuesta token a token — útil para demos en vivo.

In [ ]:
# chat_stream() en vez de chat(): en lugar de esperar a la respuesta completa,
# la API va mandando trozos de texto ("deltas") a medida que el modelo los genera.
stream = client.chat_stream(
    model=MODELO_COHERE,
    messages=[{"role": "user", "content": "Di hola en una línea, como asistente de onboarding."}],
    temperature=TEMPERATURE,
)

# Cada "event" es un fragmento de la respuesta. Solo nos interesan los de tipo
# "content-delta", que son los que traen texto nuevo; hay otros tipos de evento
# (inicio, fin, uso de tokens...) que aquí ignoramos.
for event in stream:
    if event.type == "content-delta":
        print(event.delta.message.content.text, end="", flush=True)

print()

### Hugging Face en la nube (router automático)

Mismo patrón que Cohere, pero con `InferenceClient` de `huggingface_hub` y el router de Inference Providers. Requiere token de [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) con permiso *Inference Providers*.

In [ ]:
# %pip install huggingface_hub

In [ ]:
import getpass

HF_TOKEN = getpass.getpass("Pega tu HF_TOKEN (input oculto): ").strip()
print("HF_TOKEN configurado:", "sí" if HF_TOKEN else "no")

In [ ]:
from huggingface_hub import InferenceClient

if not HF_TOKEN:
    raise ValueError("Falta HF_TOKEN. Ejecuta la celda anterior e introduce el token.")

MODELO_HF = "openai/gpt-oss-20b"  # catálogo: https://huggingface.co/inference/models
TEMPERATURE_HF = 0.2
MAX_TOKENS = 250

# A diferencia de HF local, aquí NO se descarga nada: InferenceClient solo manda
# la petición a un proveedor serverless (Groq, Together, Cerebras...) que HF elige
# automáticamente por vosotros (el "router" del que habla la guía).
hf_client = InferenceClient(api_key=HF_TOKEN)

response = hf_client.chat_completion(
    model=MODELO_HF,
    messages=[
        {"role": "user", "content": "En una frase: ¿qué es un asistente de onboarding para empleados nuevos?"}
    ],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE_HF,
)

print(f"Modelo: {MODELO_HF}")
print(response.choices[0].message.content)

¡Eso es todo por hoy! Dudas → canal de Slack de la comunidad.